In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import chromadb
client = chromadb.Client()
collection = client.create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: crud_demo
Current document count: 0


In [3]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Documents added! Total count: 5


In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

doc1 -> Python is a popular programming language for AI.
doc2 -> The weather today is sunny and warm.
doc3 -> Machine learning models learn patterns from data.


In [5]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

doc1 | {'category': 'tech'} | Python is a popular programming language for AI.
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


In [6]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc1 | distance=0.9516 | Python is a popular programming language for AI.


In [7]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc4 | distance=1.3895 | I enjoy playing football on weekends.


In [8]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

['I enjoy playing football and basketball on weekends.']
[{'updated': True, 'category': 'sports'}]


In [9]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",
        "The Eiffel Tower is located in Paris."
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

Total count after upsert: 6
['Deep learning uses neural networks with many layers and GPUs.', 'The Eiffel Tower is located in Paris.']


In [10]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['doc1', 'doc3', 'doc4', 'doc5', 'doc6']


In [11]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 2
Remaining ids: ['doc4', 'doc6']


In [12]:
final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 2
doc4 | {'category': 'sports', 'updated': True} | I enjoy playing football and basketball on weekends.
doc6 | None | The Eiffel Tower is located in Paris.


1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new one in a single call.

In [13]:
collection.add(
    documents=[
        "The sun is a star located at the center of the Solar System.",
        "Electric cars use batteries instead of gasoline.",
        "Baking requires precise measurements of ingredients.",
        "The Amazon rainforest is the largest tropical rainforest in the world.",
        "Quantum computing uses qubits instead of classical bits."
    ],
    metadatas=[
        {"category": "science"},
        {"category": "tech"},
        {"category": "cooking"},
        {"category": "nature"},
        {"category": "tech"}
    ],
    ids=["doc7", "doc8", "doc9", "doc10", "doc11"]
)

print("Added 5 new documents. Current count:", collection.count())

Added 5 new documents. Current count: 7


In [14]:
collection.upsert(
    documents=[
        "The sun is a medium-sized star at the center of our Solar System.",
        "Electric cars use advanced lithium-ion batteries instead of gasoline.",
        "Coffee is brewed from roasted coffee beans."
    ],
    metadatas=[
        {"category": "science", "updated": True},
        {"category": "tech", "updated": True},
        {"category": "beverage"}
    ],
    ids=["doc7", "doc8", "doc12"]
)

print("Upsert complete. Final document count:", collection.count())
check_results = collection.get(ids=["doc7", "doc8", "doc12"])
for doc_id, doc, meta in zip(check_results["ids"], check_results["documents"], check_results["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Upsert complete. Final document count: 8
doc7 | {'updated': True, 'category': 'science'} | The sun is a medium-sized star at the center of our Solar System.
doc8 | {'category': 'tech', 'updated': True} | Electric cars use advanced lithium-ion batteries instead of gasoline.
doc12 | {'category': 'beverage'} | Coffee is brewed from roasted coffee beans.


2. Delete all documents where `category` equals a value of your choice, then confirm with `collection.count()`.

In [15]:
collection.delete(where={"category": "cooking"})
print("Count after deleting category 'cooking':", collection.count())
print("Remaining document IDs:", collection.get()["ids"])

Count after deleting category 'cooking': 7
Remaining document IDs: ['doc4', 'doc6', 'doc7', 'doc8', 'doc10', 'doc11', 'doc12']


3. Write a small function `safe_add(collection, id, text)` that uses `upsert()` internally so it never raises a duplicate-id error.

In [16]:
def safe_add(collection, id, text, metadata=None):
    """
    Safely adds a document to the collection using upsert,
    preventing duplicate key errors.
    """
    collection.upsert(
        ids=[id],
        documents=[text],
        metadatas=[metadata] if metadata else [None]
    )
safe_add(collection, "doc13", "This is a completely new safe document.", {"category": "safety"})
print("doc13 state:", collection.get(ids=["doc13"])["documents"])
safe_add(collection, "doc13", "This is an updated version of the safe document.", {"category": "safety", "updated": True})
print("doc13 state after update:", collection.get(ids=["doc13"])["documents"])
print("Metadata after update:", collection.get(ids=["doc13"])["metadatas"])

doc13 state: ['This is a completely new safe document.']
doc13 state after update: ['This is an updated version of the safe document.']
Metadata after update: [{'category': 'safety', 'updated': True}]
